# Simulation Benchmark Validation

Objective:
- Validate the new held-out simulation benchmark protocol locally without SLURM.
- Confirm shared cached train/test splits, challenge-region membership, held-out posterior metrics, simulated PPC, benchmark launcher, and aggregation.

Success criteria:
- Training samples are outside the held-out joint region and held-out test samples are inside it.
- Held-out test data is stable across methods and training seeds for the same benchmark cell.
- `metrics.json` contains `heldout_test_posterior_vs_true`, `heldout_test_ppc`, `heldout_test_stats`, and `budget_metadata`.
- Local smoke and quick runs complete for `npe`, `npse`, `fnpe`, and `simformer` on CPU.
- A shared-data quick comparison completes for `npe`, `npse`, and `simformer`.
- The benchmark launcher and aggregation script both produce real outputs on a tiny repeated-seed study.

Notes:
- This notebook intentionally uses `--device cpu` because the local machine has no CUDA device.
- The local Python environment prints NumPy binary-compatibility warnings from optional dependencies. Those warnings are noisy but non-fatal for the validation flow below.


In [1]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'scripts' / 'run_simulation_comparison.py').exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
assert (REPO_ROOT / 'scripts' / 'run_simulation_comparison.py').exists(), REPO_ROOT
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
TMP_LOG_DIR = REPO_ROOT / 'tmp' / 'notebook_logs'
TMP_LOG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cpu'
QUICK_NUM_SIMS = 64
QUICK_TSEG = 128
SMOKE_NUM_SIMS = 64
SMOKE_TSEG = 128
NBAGG_GROUP = f"nbagg_{int(time.time())}"
PYTHON = sys.executable


def _slug(text: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', text).strip('_')


def run_cmd(cmd: list[str], label: str, timeout: int = 1800, tail_lines: int = 40) -> dict:
    log_path = TMP_LOG_DIR / f"{_slug(label)}.log"
    started = time.time()
    proc = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    duration = time.time() - started
    combined = (proc.stdout or '') + ('\n' + proc.stderr if proc.stderr else '')
    log_path.write_text(combined, encoding='utf-8')
    tail = '\n'.join(combined.splitlines()[-tail_lines:])
    print(f'[{label}] exit={proc.returncode} duration={duration:.1f}s log={log_path}')
    print(tail)
    if proc.returncode != 0:
        raise RuntimeError(f'{label} failed with exit code {proc.returncode}')
    return {'label': label, 'duration_s': duration, 'log_path': str(log_path), 'tail': tail}


def load_json(path: str | Path):
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)


print({'repo_root': str(REPO_ROOT), 'python': PYTHON, 'torch_cuda_available': torch.cuda.is_available()})


{'repo_root': 'C:\\Users\\aritr\\Documents\\thesis-sbi-aritra\\code', 'python': 'C:\\Users\\aritr\\anaconda3\\python.exe', 'torch_cuda_available': False}


## Benchmark Validation Configuration

The notebook uses the smallest local settings that still exercise the full benchmark path:
- shared cached training split
- shared cached held-out test split
- held-out posterior metrics and statistical testing
- simulated PPC on held-out synthetic cases
- local quick runs on CPU for all four methods
- one repeated-seed benchmark group for aggregation


In [2]:
from configs.config import ExperimentConfig
from inference.unified_experiment import (
    _joint_holdout_mask,
    get_or_generate_test_dataset,
    get_or_generate_training_dataset,
)
from models.models import build_prior
from simulation.simulation import init_simulation_from_config, make_simulator
from utils.env_utils import get_device, setup_environment

base_cfg = ExperimentConfig(
    exp_name='nb_split_validation',
    method='npe',
    num_simulations=QUICK_NUM_SIMS,
    T_seg=QUICK_TSEG,
    active_parameters=('mu',),
    device=DEVICE,
    random_seed=42,
    sim_seed=42,
    train_seed=43,
    cache_dataset=True,
    reuse_dataset=True,
    run_simulated_test_eval=True,
    run_simulated_ppc=True,
    num_test_simulations=10,
    no_plots=True,
)

setup_environment(base_cfg.random_seed)
device = get_device(DEVICE)
init_simulation_from_config(base_cfg)
prior = build_prior(base_cfg, device)
simulator = make_simulator(base_cfg, device)

theta_train, x_train, train_region_meta = get_or_generate_training_dataset(base_cfg, prior, simulator, device)
theta_test, x_test, test_region_meta = get_or_generate_test_dataset(base_cfg, prior, simulator, device)

train_inside = int(_joint_holdout_mask(base_cfg, theta_train, x_train, train_region_meta).sum().item())
test_inside = int(_joint_holdout_mask(base_cfg, theta_test, x_test, test_region_meta).sum().item())

split_summary = {
    'dataset_id': base_cfg.dataset_id,
    'test_dataset_id': base_cfg.test_dataset_id,
    'theta_train_shape': tuple(theta_train.shape),
    'x_train_shape': tuple(x_train.shape),
    'theta_test_shape': tuple(theta_test.shape),
    'x_test_shape': tuple(x_test.shape),
    'train_inside_holdout': train_inside,
    'test_inside_holdout': test_inside,
}
assert train_inside == 0, split_summary
assert test_inside == int(theta_test.shape[0]), split_summary
split_summary



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\aritr\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\aritr\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\aritr\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "C:\Users\aritr\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\aritr\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\aritr\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\aritr\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "C:\Users\aritr\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Torch device: cpu | CUDA available: False CUDA version: None
JAX backend: cpu
No GPU available; running on CPU.
[DATA] Loading cached dataset from datasets\dataset_bc194fd6bdc2.pt
[DATA] Loading cached held-out test dataset from datasets\dataset_test_c541d6072cd3.pt


C:\Users\aritr\Documents\thesis-sbi-aritra\code\inference\unified_experiment.py:376: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cached = torch.load(cache_path)
C:\Users\a

{'dataset_id': 'dataset_bc194fd6bdc2',
 'test_dataset_id': 'dataset_test_c541d6072cd3',
 'theta_train_shape': (64, 1),
 'x_train_shape': (64, 128, 13),
 'theta_test_shape': (10, 1),
 'x_test_shape': (10, 128, 13),
 'train_inside_holdout': 0,
 'test_inside_holdout': 10}

In [3]:
cfg_same_eval = ExperimentConfig(
    exp_name='nb_split_validation_alt',
    method='npse',
    num_simulations=QUICK_NUM_SIMS,
    T_seg=QUICK_TSEG,
    active_parameters=('mu',),
    device=DEVICE,
    random_seed=43,
    sim_seed=43,
    train_seed=44,
    cache_dataset=True,
    reuse_dataset=True,
    run_simulated_test_eval=True,
    run_simulated_ppc=True,
    num_test_simulations=10,
    benchmark_eval_seed=base_cfg.benchmark_eval_seed,
    no_plots=True,
)
assert cfg_same_eval.test_dataset_id == base_cfg.test_dataset_id

setup_environment(cfg_same_eval.random_seed)
prior_same = build_prior(cfg_same_eval, device)
init_simulation_from_config(cfg_same_eval)
simulator_same = make_simulator(cfg_same_eval, device)
theta_test_same, x_test_same, _ = get_or_generate_test_dataset(cfg_same_eval, prior_same, simulator_same, device)
assert torch.equal(theta_test, theta_test_same)
assert torch.equal(x_test, x_test_same)

cfg_region_changed = ExperimentConfig(
    exp_name='nb_split_validation_changed',
    method='npe',
    num_simulations=QUICK_NUM_SIMS,
    T_seg=QUICK_TSEG,
    active_parameters=('mu',),
    device=DEVICE,
    random_seed=42,
    sim_seed=42,
    train_seed=43,
    cache_dataset=True,
    reuse_dataset=True,
    run_simulated_test_eval=True,
    run_simulated_ppc=True,
    num_test_simulations=10,
    test_region_min_brake=250.0,
    no_plots=True,
)
assert cfg_region_changed.dataset_id != base_cfg.dataset_id
assert cfg_region_changed.test_dataset_id != base_cfg.test_dataset_id

{
    'shared_test_dataset_id_across_seeds': base_cfg.test_dataset_id,
    'changed_region_dataset_id': cfg_region_changed.dataset_id,
    'changed_region_test_dataset_id': cfg_region_changed.test_dataset_id,
}


[DATA] Loading cached held-out test dataset from datasets\dataset_test_c541d6072cd3.pt


{'shared_test_dataset_id_across_seeds': 'dataset_test_c541d6072cd3',
 'changed_region_dataset_id': 'dataset_c9f7d4a86768',
 'changed_region_test_dataset_id': 'dataset_test_cadd86e8eef4'}

## Smoke Sanity

Smoke mode should still work as a minimal orchestration check while skipping the heavy held-out evaluation path.


In [4]:
smoke_cmd = [
    PYTHON,
    'scripts/run_simulation_comparison.py',
    '--exp-name', 'nb_smoke_mu',
    '--num-simulations', str(SMOKE_NUM_SIMS),
    '--T-seg', str(SMOKE_TSEG),
    '--methods', 'npe', 'npse', 'simformer',
    '--seed', '42',
    '--params', 'mu',
    '--smoke',
    '--device', DEVICE,
]
smoke_result = run_cmd(smoke_cmd, 'smoke_mu', timeout=600)
smoke_summary = load_json(REPO_ROOT / 'experiments' / 'nb_smoke_mu_summary.json')
sorted(smoke_summary.keys())


[smoke_mu] exit=0 duration=18.2s log=C:\Users\aritr\Documents\thesis-sbi-aritra\code\tmp\notebook_logs\smoke_mu.log
  File "C:\Users\aritr\anaconda3\Lib\site-packages\sklearn\base.py", line 19, in <module>
    from .utils._metadata_requests import _MetadataRequester, _routing_enabled
  File "C:\Users\aritr\anaconda3\Lib\site-packages\sklearn\utils\__init__.py", line 9, in <module>
    from ._chunking import gen_batches, gen_even_slices
  File "C:\Users\aritr\anaconda3\Lib\site-packages\sklearn\utils\_chunking.py", line 11, in <module>
    from ._param_validation import Interval, validate_params
  File "C:\Users\aritr\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 17, in <module>
    from .validation import _is_arraylike_not_scalar
  File "C:\Users\aritr\anaconda3\Lib\site-packages\sklearn\utils\validation.py", line 21, in <module>
    from ..utils._array_api import _asarray_with_order, _is_numpy_namespace, get_namespace
  File "C:\Users\aritr\anaconda3\Lib\site-p

['npe', 'npse', 'simformer']

## Quick Single-Method End-to-End Checks

These runs validate training plus held-out diagnostics for each method on CPU. The FNPE path uses a reduced local CPU quick budget so the notebook remains runnable.


In [5]:
single_method_results = {}
for method, timeout in [('npe', 600), ('npse', 900), ('simformer', 600), ('fnpe', 1200)]:
    exp_name = f'nb_quick_{method}_mu'
    cmd = [
        PYTHON,
        'scripts/run_simulation_comparison.py',
        '--exp-name', exp_name,
        '--num-simulations', str(QUICK_NUM_SIMS),
        '--T-seg', str(QUICK_TSEG),
        '--methods', method,
        '--seed', '42',
        '--params', 'mu',
        '--quick',
        '--device', DEVICE,
    ]
    run_cmd(cmd, f'quick_{method}', timeout=timeout)
    summary = load_json(REPO_ROOT / 'experiments' / f'{exp_name}_summary.json')
    assert method in summary
    result = summary[method]
    posterior_metrics = result.get('posterior_vs_true') or result.get('heldout_test_posterior_vs_true') or {}
    heldout_ppc = result.get('heldout_test_ppc') or {}
    assert posterior_metrics
    assert heldout_ppc
    single_method_results[method] = {
        'train_time_s': result.get('train_time_s'),
        'w2_mean': posterior_metrics.get('w2_mean'),
        'ppc_rmse_mean': heldout_ppc.get('rmse_mean', heldout_ppc.get('aggregate', {}).get('rmse_mean')),
    }

single_method_results


[quick_npe] exit=0 duration=14.5s log=C:\Users\aritr\Documents\thesis-sbi-aritra\code\tmp\notebook_logs\quick_npe.log
100%|##########| 10/10 [00:00<00:00, 834.12it/s]

100%|##########| 10/10 [00:00<00:00, 714.23it/s]

  0%|          | 0/10 [00:00<?, ?it/s]
109it [00:00, 4359.69it/s]            

100%|##########| 20/20 [00:00<00:00, 1532.42it/s]

100%|##########| 20/20 [00:00<00:00, 1538.38it/s]

100%|##########| 20/20 [00:00<00:00, 1476.01it/s]

  0%|          | 0/20 [00:00<?, ?it/s]
118it [00:00, 4624.24it/s]            

  0%|          | 0/20 [00:00<?, ?it/s]
119it [00:00, 4660.69it/s]            

100%|##########| 20/20 [00:00<00:00, 1538.04it/s]

100%|##########| 20/20 [00:00<00:00, 1324.31it/s]

100%|##########| 20/20 [00:00<00:00, 1478.12it/s]

100%|##########| 20/20 [00:00<00:00, 1538.69it/s]

100%|##########| 20/20 [00:00<00:00, 1669.74it/s]

100%|##########| 10/10 [00:00<00:00, 796.84it/s]


[quick_npse] exit=0 duration=124.9s log=C:\Users\aritr\Documents\thesis-sbi-aritra\code\tmp\notebook_logs\quick_npse.log

Generating 10 posterior samples in 499 diffusion steps.:  65%|######5   | 325/499 [00:03<00:01, 90.93it/s]

Generating 10 posterior samples in 499 diffusion steps.:  67%|######7   | 335/499 [00:03<00:01, 91.67it/s]

Generating 10 posterior samples in 499 diffusion steps.:  69%|######9   | 345/499 [00:03<00:01, 91.78it/s]

Generating 10 posterior samples in 499 diffusion steps.:  71%|#######1  | 355/499 [00:03<00:01, 92.45it/s]

Generating 10 posterior samples in 499 diffusion steps.:  73%|#######3  | 365/499 [00:03<00:01, 94.17it/s]

Generating 10 posterior samples in 499 diffusion steps.:  75%|#######5  | 375/499 [00:03<00:01, 95.35it/s]

Generating 10 posterior samples in 499 diffusion steps.:  77%|#######7  | 385/499 [00:04<00:01, 95.72it/s]

Generating 10 posterior samples in 499 diffusion steps.:  79%|#######9  | 395/499 [00:04<00:01, 95.24it/s]

Generating 10 

[quick_simformer] exit=0 duration=23.7s log=C:\Users\aritr\Documents\thesis-sbi-aritra\code\tmp\notebook_logs\quick_simformer.log
  File "C:\Users\aritr\anaconda3\Lib\site-packages\pandas\__init__.py", line 49, in <module>
    from pandas.core.api import (
  File "C:\Users\aritr\anaconda3\Lib\site-packages\pandas\core\api.py", line 9, in <module>
    from pandas.core.dtypes.dtypes import (
  File "C:\Users\aritr\anaconda3\Lib\site-packages\pandas\core\dtypes\dtypes.py", line 24, in <module>
    from pandas._libs import (
  File "C:\Users\aritr\anaconda3\Lib\site-packages\pyarrow\__init__.py", line 65, in <module>
    import pyarrow.lib as _lib
Traceback (most recent call last):
  File "C:\Users\aritr\anaconda3\Lib\site-packages\numpy\core\_multiarray_umath.py", line 46, in __getattr__
    raise ImportError(msg)
ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled wi

[quick_fnpe] exit=0 duration=170.7s log=C:\Users\aritr\Documents\thesis-sbi-aritra\code\tmp\notebook_logs\quick_fnpe.log
    from pandas.core.dtypes.dtypes import (
  File "C:\Users\aritr\anaconda3\Lib\site-packages\pandas\core\dtypes\dtypes.py", line 24, in <module>
    from pandas._libs import (
  File "C:\Users\aritr\anaconda3\Lib\site-packages\pyarrow\__init__.py", line 65, in <module>
    import pyarrow.lib as _lib
Traceback (most recent call last):
  File "C:\Users\aritr\anaconda3\Lib\site-packages\numpy\core\_multiarray_umath.py", line 46, in __getattr__
    raise ImportError(msg)
ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that som

{'npe': {'train_time_s': 1.998403549194336,
  'w2_mean': 0.4014364182949066,
  'ppc_rmse_mean': 0.7226495742797852},
 'npse': {'train_time_s': 4.55097770690918,
  'w2_mean': 0.3561180531978607,
  'ppc_rmse_mean': 0.7125217914581299},
 'simformer': {'train_time_s': 6.423529624938965,
  'w2_mean': 0.35010239481925964,
  'ppc_rmse_mean': 0.7125625610351562},
 'fnpe': {'train_time_s': 4.819936752319336,
  'w2_mean': 0.6789376735687256,
  'ppc_rmse_mean': 0.7220756411552429}}

## Shared-Data Quick Comparison

This run checks that the authoritative shared-data comparison path still works for the strictly shared-data methods.


In [6]:
compare_cmd = [
    PYTHON,
    'scripts/run_simulation_comparison.py',
    '--exp-name', 'nb_quick_compare_mu',
    '--num-simulations', str(QUICK_NUM_SIMS),
    '--T-seg', str(QUICK_TSEG),
    '--methods', 'npe', 'npse', 'simformer',
    '--seed', '42',
    '--params', 'mu',
    '--quick',
    '--device', DEVICE,
]
run_cmd(compare_cmd, 'quick_compare', timeout=1200)
compare_summary = load_json(REPO_ROOT / 'experiments' / 'nb_quick_compare_mu_summary.json')
comparison_table = sorted([
    {
        'method': method,
        'w2_mean': (result.get('posterior_vs_true') or result.get('heldout_test_posterior_vs_true') or {}).get('w2_mean'),
        'coverage_90': (result.get('posterior_vs_true') or result.get('heldout_test_posterior_vs_true') or {}).get('coverage_90'),
        'ppc_rmse_mean': result.get('heldout_test_ppc', {}).get('rmse_mean', result.get('heldout_test_ppc', {}).get('aggregate', {}).get('rmse_mean')),
    }
    for method, result in compare_summary.items()
], key=lambda row: row['method'])
comparison_table


[quick_compare] exit=0 duration=144.7s log=C:\Users\aritr\Documents\thesis-sbi-aritra\code\tmp\notebook_logs\quick_compare.log

Generating 10 posterior samples in 499 diffusion steps.:  82%|########2 | 411/499 [00:04<00:00, 99.41it/s] 

Generating 10 posterior samples in 499 diffusion steps.:  84%|########4 | 421/499 [00:04<00:00, 98.90it/s]

Generating 10 posterior samples in 499 diffusion steps.:  86%|########6 | 431/499 [00:04<00:00, 97.77it/s]

Generating 10 posterior samples in 499 diffusion steps.:  89%|########8 | 442/499 [00:04<00:00, 99.10it/s]

Generating 10 posterior samples in 499 diffusion steps.:  91%|######### | 453/499 [00:04<00:00, 99.99it/s]

Generating 10 posterior samples in 499 diffusion steps.:  93%|#########2| 463/499 [00:04<00:00, 99.76it/s]

Generating 10 posterior samples in 499 diffusion steps.:  95%|#########4| 474/499 [00:04<00:00, 100.76it/s]

Generating 10 posterior samples in 499 diffusion steps.:  97%|#########7| 485/499 [00:04<00:00, 101.04it/s]

Gener

[{'method': 'npe',
  'w2_mean': 0.4014364182949066,
  'coverage_90': 0.2,
  'ppc_rmse_mean': 0.7226495742797852},
 {'method': 'npse',
  'w2_mean': 0.3561180531978607,
  'coverage_90': 0.1,
  'ppc_rmse_mean': 0.7125217914581299},
 {'method': 'simformer',
  'w2_mean': 0.3431185483932495,
  'coverage_90': 0.0,
  'ppc_rmse_mean': 0.7098314166069031}]

## Tiny Repeated-Seed Benchmark and Aggregation

Use the benchmark launcher locally with two seeds, then aggregate the resulting experiment directories.


In [7]:
launch_cmd = [
    PYTHON,
    'scripts/launch_simulation_benchmark.py',
    '--group-name', NBAGG_GROUP,
    '--num-simulations', str(QUICK_NUM_SIMS),
    '--methods', 'npe', 'simformer',
    '--params-grid', 'mu',
    '--tseg-grid', str(QUICK_TSEG),
    '--seeds', '42', '43',
    '--mode', 'local',
    '--run-mode', 'yes',
    '--device', DEVICE,
]
run_cmd(launch_cmd, 'launch_nbagg', timeout=1800)

aggregate_cmd = [
    PYTHON,
    'scripts/aggregate_simulation_benchmark.py',
    '--experiments-root', 'experiments',
    '--exp-prefix', NBAGG_GROUP,
    '--output-json', f'experiments/{NBAGG_GROUP}_aggregate.json',
    '--output-csv', f'experiments/{NBAGG_GROUP}_aggregate.csv',
]
run_cmd(aggregate_cmd, 'aggregate_nbagg', timeout=300)
aggregate = load_json(REPO_ROOT / 'experiments' / f'{NBAGG_GROUP}_aggregate.json')
assert aggregate['num_runs'] == 4
assert len(aggregate['aggregate_rows']) == 2
assert any(pair['pvalue'] is not None for pair in aggregate['pairwise_tests'])
sorted(
    [
        {
            'method': row['method'],
            'params': row['params'],
            'T_seg': row['T_seg'],
            'num_runs': row['num_runs'],
            'total_budget_steps': row['total_budget_steps'],
        }
        for row in aggregate['aggregate_rows']
    ],
    key=lambda row: (row['params'], row['T_seg'], row['method']),
)


[launch_nbagg] exit=0 duration=58.5s log=C:\Users\aritr\Documents\thesis-sbi-aritra\code\tmp\notebook_logs\launch_nbagg.log
116it [00:00, 4726.67it/s]            

100%|##########| 20/20 [00:00<00:00, 1539.19it/s]

100%|##########| 20/20 [00:00<00:00, 1543.87it/s]

100%|##########| 20/20 [00:00<00:00, 1593.58it/s]

  0%|          | 0/20 [00:00<?, ?it/s]
118it [00:00, 4261.66it/s]            

100%|##########| 20/20 [00:00<00:00, 1538.12it/s]

  0%|          | 0/20 [00:00<?, ?it/s]
115it [00:00, 4239.13it/s]            

100%|##########| 10/10 [00:00<00:00, 724.45it/s]
C:\Users\aritr\Documents\thesis-sbi-aritra\code\inference\unified_experiment.py:444: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more deta

[aggregate_nbagg] exit=0 duration=0.9s log=C:\Users\aritr\Documents\thesis-sbi-aritra\code\tmp\notebook_logs\aggregate_nbagg.log
Wrote experiments\nbagg_1773978777_aggregate.json
Wrote experiments\nbagg_1773978777_aggregate.csv


[{'method': 'npe',
  'params': 'mu',
  'T_seg': 128,
  'num_runs': 2,
  'total_budget_steps': 8192},
 {'method': 'simformer',
  'params': 'mu',
  'T_seg': 128,
  'num_runs': 2,
  'total_budget_steps': 8192}]

## Final Checks and Next Steps

This notebook confirmed:
- held-out train/test split generation and membership constraints
- deterministic held-out test reuse across methods and seeds
- held-out posterior metrics and simulated PPC for all methods
- shared-data comparison for `npe`, `npse`, and `simformer`
- benchmark launcher and aggregation outputs on a tiny repeated-seed benchmark

Next steps before ablations:
- move to cluster-backed full runs using the validated commands
- keep encoder benchmarking deferred until the baseline benchmark matrix is complete
- optionally clean up the local Python environment warnings by aligning NumPy and compiled dependency versions
